# MODULE 43 — OBJECT SEGMENTATION

# 6.4 Mask R-CNN — Instance Segmentation

**Must Know · Instance Segmentation**

We've studied three important semantic segmentation architectures:

1. FCN — Fully Convolutional Network.
2. U-Net — Encoder–Decoder with Skip Connections.
3. DeepLab — Atrous Convolution and ASPP.

Now let's study Mask R-CNN, which approaches segmentation differently.

Mask R-CNN performs object detection and instance segmentation together.

Its key idea is to identify individual objects, locate them using bounding boxes, and predict a separate pixel-level mask for each object.

---

## 1. Why do we need Mask R-CNN?

Imagine an image containing three cars.

![Mask R-CNN instance segmentation example](https://images.openai.com/static-rsc-4/Hh9Z4o864RPLcj5B0UHhWwsyL316Y0TxANVQBsCeSouGbFNFl9CVGTDxWwYsWrUWuHL-6W3kpbDfYv8P3_YXpiCxeMiEUw9K8LFWoU9tEZ_yPhBtjz7hnZtEOhtaQ-lcvByb4QOB3QWJMQ5P1H6vEz5NvqdrMyREt8bUKFXhE6o?purpose=inline)

Source: [Mask R-CNN — CSDN](https://blog.csdn.net/ghw15221836342/article/details/80084861)

Consider the three segmentation approaches:

**Semantic segmentation:**

- Identifies all car pixels as `Car`.
- Cannot distinguish individual cars.

**Instance segmentation:**

- Identifies Car 1.
- Identifies Car 2.
- Identifies Car 3.
- Produces separate masks for each car.

Mask R-CNN solves this instance-level segmentation problem.

### What is Mask R-CNN?

Mask R-CNN is an extension of Faster R-CNN.

Faster R-CNN performs object detection using:

- Object classification.
- Bounding-box regression.

Mask R-CNN adds another branch:

- Pixel-level mask prediction.

![Mask R-CNN medical image example](https://images.openai.com/static-rsc-4/NRFo2XiD1l_ZJuLyois7AT3bkq8YXIqPAgf3vTvftWGm9ehwemM3k7VYk_WKQD5hxPvKbLo7a7W0xf1HjheFloQNDMYzsDzAtZoLLbUAhPS-Wx3Xjrqw1Yl8zLuN_3Nc-4NnDyUT5fgxWNBBIxpVpdKs_u4xXOdQ5bxqgqHuuts?purpose=inline)

Source: [Deep Learning-Based Dental Caries Diagnosis — PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC12985255/)

This gives us three outputs for each detected object:

1. Class label — What is the object?
2. Bounding box — Where is the object?
3. Segmentation mask — Which pixels belong to the object?

---

## 2. Overall Mask R-CNN Architecture

![Mask R-CNN architecture overview](https://images.openai.com/static-rsc-4/HnOfdkDPpGcxGhnm7kHw2u-O5IyTuXQNNdK6eoXZaU39hUH5Lvh1RFUu2K9ZGoRGjsuxeitCyEruwdST-oOYe5RLA6jb2JcQ1SSf4rBEDm0tnqHX7rcnaFILOXUihHfKOiZUnUdIgZDJKNjb18mLvWjU7T-eDjk5HTzUaHwJkw8?purpose=inline)

Source: [Distributed Mask R-CNN Training with Amazon SageMaker](https://aws.amazon.com/blogs/machine-learning/distributed-mask-rcnn-training-with-amazon-sagemakercv/)

Understand these six components in order:

1. Backbone network.
2. Feature Pyramid Network (FPN).
3. Region Proposal Network (RPN).
4. ROI Align.
5. Classification and bounding-box regression branches.
6. Mask prediction branch.

Let's break them down.

## 3. Backbone — Feature Extraction

**Stage 1**

The first component is the backbone CNN.

Its responsibility is to extract meaningful features from the input image.

Common backbone architectures include:

- ResNet-50.
- ResNet-101.

![CNN feature extraction](https://images.openai.com/static-rsc-4/CmNtrelvj98Clazt3yZPQDZW-4gJhV0mHh_mln65G66Fn7mzo_856M8h4Tur9VOgW6zabBrwm719bMCiBpObX14AWXYGaJwCivX2d3ExSQVzZGjgkKtiMFBDmrf4Nq8BzymbOE6M_IfwUF5rXVQ9jj2Kmaw8IXxGpB_s-2VsFdM?purpose=inline)

Source: [Image Recognition and Common Algorithms — Cloudinary](https://cloudinary.com/guides/front-end-development/image-recognition-machine-learning-use-cases-and-common-algorithms)

For example:

```text
Input Image
     ↓
Convolution Layers
     ↓
Residual Blocks
     ↓
Feature Maps
```

Suppose the input image is:

```text
[B, 3, 800, 800]
```

The backbone generates feature maps at different spatial resolutions.

These feature maps contain information about:

- Edges and textures.
- Shapes.
- Object parts.
- High-level object representations.

**Connection to previous topics:** Just like the encoder in U-Net, the backbone extracts increasingly meaningful visual features.

---

## 4. Feature Pyramid Network (FPN)

**Stage 2**

Why do we need FPN?

Imagine detecting:

- A large bus.
- A medium-sized car.
- A small traffic sign.

Using only one feature-map resolution can make detection difficult across different object sizes.

FPN addresses this through multi-scale feature extraction and fusion.

![Feature Pyramid Network architecture](https://images.openai.com/static-rsc-4/uYOk51KaMp8oSm_4yjxFKHg0KO1hyCxIXVHJgg5kBiUQHGAYDy8PIXaipu25zXVSDEM5_MdzHUNBsW3W35s-kWmW8xzHPQ1EWBTxtSAuILrUW2z-x3r7gtWlhPIajK7b2gWLE8Mo19xuQxauax1m0VPXERA-ogudf81H3SKIITY?purpose=inline)

Source: [FPN — Velog](https://velog.io/%40snoop2head/posts)

### How does FPN work?

It combines two pathways.

**1. Bottom-up pathway**

This is the normal CNN backbone.

- Extracts increasingly deep features.
- Spatial resolution decreases.
- Semantic information becomes stronger.

**2. Top-down pathway**

- Starts with deeper feature maps.
- Upsamples them.
- Combines them with earlier backbone features through lateral connections.

This creates feature maps at multiple resolutions.

Conceptually:

```text
Backbone
   |
   ├── C2
   ├── C3
   ├── C4
   └── C5
        |
        ↓
Top-down pathway
        |
        ↓
   P2, P3, P4, P5
```

These pyramid feature maps help the detection system handle objects of different sizes.

**Remember:** FPN improves multi-scale feature representation. It is commonly used with Mask R-CNN, but it is not the mask-prediction branch itself.

---

## 5. RPN — Region Proposal Network

<text>**Extremely Important**</text>

The backbone and FPN extract features, but we still need to identify where objects might exist.

This is the responsibility of the Region Proposal Network (RPN).

### What is a region proposal?

A region proposal is a candidate image region that may contain an object.

For example, an image might contain:

- A car in the top-left region.
- A pedestrian in the centre.
- A bicycle on the right.

RPN generates candidate bounding boxes around possible objects.

![Region Proposal Network illustration](https://images.openai.com/static-rsc-4/3_nT3bPH_tgHjjLiyiO7cWMnALDnNyA5qEB6RzGsTlGhs4C9FQPBVGAIXR5FGbyWThA0Drfd6MRZgWUoXAhzpeimPXe2NT4WqD-_HYjr3D5xJ5CnZ7VjwWXLph6UK5Tc-YBpSyZ_taO76yjvXTW3B7vqXXbcOGpEnYYPACVN1p4?purpose=inline)

Source: [Object Detection and Segmentation — Aman AI Journal](https://aman.ai/cs231n/detection/)

### How does RPN work?

RPN operates on the feature maps produced by FPN.

It uses a small sliding convolutional network to predict two things at different spatial positions:

**1. Objectness score**

Estimates whether an anchor contains an object or background.

```text
Anchor 1 → Object: 0.95
Anchor 2 → Object: 0.80
Anchor 3 → Object: 0.10
```

**2. Bounding-box regression**

Predicts adjustments to anchor coordinates so that candidate boxes better fit objects.

### What are anchors?

Anchors are predefined reference boxes placed at different spatial positions, scales and aspect ratios.

![Anchor boxes in Faster R-CNN](https://images.openai.com/static-rsc-4/XQ5AjNlbVuUdUQZFP26piXRXrFfupm9TJXbjMtmLAdmeINULGpx4yYz1mFC6LkoAEKxzzEGz1SzK93NP6VKmee2dVZ9yZCbcB5-KTsSALtO4qh1jJLUdYdDd0qmapZrfDLwtDbthChFoMG2tjlmAFr5en6KXH6TJ-DzeERBx-Cg?purpose=inline)

Source: [Faster R-CNN Anchor Boxes — Data Science Stack Exchange](https://datascience.stackexchange.com/questions/27277/faster-rcnn-how-anchor-work-with-slider-in-rpn-layer)

For example:

- Small square anchor.
- Large square anchor.
- Wide rectangular anchor.
- Tall rectangular anchor.

RPN predicts objectness and coordinate adjustments for these anchors.

### RPN workflow

```text
FPN Feature Maps
       ↓
Sliding Convolution
       ↓
Anchor Boxes
       ↓
Objectness Prediction
       +
Bounding Box Regression
       ↓
Candidate Proposals
       ↓
Non-Maximum Suppression
       ↓
Final Region Proposals
```

**What is Non-Maximum Suppression (NMS)?**

Suppose RPN generates five overlapping boxes around the same car.

NMS removes redundant overlapping proposals while retaining the strongest candidates.

This reduces duplicate detections.

**Important distinction:** RPN proposes object regions. It does not yet determine the final object class or generate the final segmentation mask.

---

## 6. ROI Align — Region of Interest Align

<text>**Extremely Important**</text>

We now have region proposals.

But each proposal can have a different size.

For example:

```text
Proposal 1: 100 × 80
Proposal 2: 50 × 120
Proposal 3: 200 × 150
```

The prediction heads need feature representations with consistent dimensions.

This is where ROI Align is used.

### What is ROI Align?

ROI Align extracts features corresponding to a proposed object region and converts them into a fixed spatial size.

![ROI Align illustration](https://images.openai.com/static-rsc-4/9FA-grwZWLpGe0N_dGCgT9-5cw5I7JD2Qa3fFcCtwjdVhUQ5EWvwMuWybl0-DvcIQq-XWR93wpUOg--a7WqydlBcRzb05Oxq5BnehjrDNUyx5JlFKue0Pjab2Vh6jJtNS77ew2zVRur3Z0qy464j4aeZgjRUHAEnHAub8IIU9a0?purpose=inline)

Source: [Detection and Segmentation — Young's Notes](https://younghk.github.io/machine-learning/2019-11-19---cs231n-lec12-detection-and-segmentation/)

For example:

```text
ROI 1 → 7 × 7 feature map
ROI 2 → 7 × 7 feature map
ROI 3 → 7 × 7 feature map
```

All proposed regions can now be processed by the subsequent prediction heads.

### Why not ROI Pooling?

Earlier Faster R-CNN used ROI Pooling.

ROI Pooling uses coordinate quantization, which can introduce small spatial misalignments.

This is particularly problematic for segmentation because accurate object boundaries matter.

ROI Align improves spatial alignment by avoiding this quantization and using bilinear interpolation to sample feature values at precisely defined positions.

![ROI Pooling vs ROI Align](https://images.openai.com/static-rsc-4/jg3ijPP2_U98uvh-9hSM0ZyTLBf5x23N7nU0I_v2MJqJJbMCm4YNJRmzSa_OXewVaXu9Fd2ZBJr7hA35ewfv571_5PLBTfvOFvjGBA-tBh2HhavaOSNAuSEuP7ZpN1v6EoqruWfQCgIlcSVcJTyumRyh6FxKmSv0uioHpmb3p6I?purpose=inline)

Source: [ROI Pooling and ROI Align — CSDN](https://blog.csdn.net/lx_ros/article/details/126335242)

### Bilinear interpolation — Basic intuition

Suppose a sampling location falls between four feature-map values.

Bilinear interpolation estimates the value using the distances to those four neighbouring positions.

This allows ROI Align to preserve more accurate spatial correspondence.

**Why is it important?**

A small positional error might be acceptable for a bounding box, but it can produce noticeable errors in a pixel-level mask.

That is why ROI Align is especially important for Mask R-CNN.

---

## 7. Three Prediction Branches

<text>**Core Mask R-CNN Concept**</text>

After ROI Align, the extracted object features are passed to three prediction branches.

![Mask R-CNN three prediction branches](https://images.openai.com/static-rsc-4/714DhvJPyNm8Lyuy-_oaQLqbmpo54OcS_kNl3OqFyWdo6ACMPmlBVMh7QHwDJ3NsvK0k3u-5EVW25EtwG87q1apkhxG8sYB-871vEoGVpHEo2Im_q98MHq2m7P8H7ZPAunBFs_q2f-A8gIBzEFk7GFinGSLFUlUaDkt0bnSjT-o?purpose=inline)

Source: [Mask R-CNN — He, Gkioxari, Dollár and Girshick (2017)](https://www.researchgate.net/figure/Mask-R-CNN-He-Gkioxari-Dollar-and-Girshick-2017_fig1_360962432)

### Branch 1 — Classification

This branch predicts the object's semantic category.

Example:

```text
ROI Features
     ↓
Classification Head
     ↓
Car:        0.90
Person:     0.05
Bicycle:    0.03
Background: 0.02
```

Final prediction:

```text
Class = Car
```

### Branch 2 — Bounding-box Regression

This branch refines the bounding-box coordinates.

For example:

```text
Initial Proposal:
(x1, y1, x2, y2)

       ↓

Bounding-box Regression

       ↓

Refined Bounding Box:
(x1', y1', x2', y2')
```

The goal is to fit the object more accurately.

### Branch 3 — Mask Prediction

<text>**The Main Extension Over Faster R-CNN**</text>

This is the additional branch introduced by Mask R-CNN.

It predicts a binary mask for each object proposal.

![Mask prediction branch](https://images.openai.com/static-rsc-4/m3M_MnZ3pezf3gAtT8hfdLrtRzn4evApUDC6hltC4ozfAbQehCM_QLa3SS-mj283YHoxpC-61t5ZOvwB4qIoAxOVbFgkHyDhPz5g6P7wc039980AnE86K8CsylNurVrsNTEckdgKggBx4KbV8rihtMf2uOoGx8-Sa-cWv88aIIc?purpose=inline)

Source: [Mask R-CNN — Lee's Hub](https://lee-vius.github.io/cv/ml/2022/02/15/blog-MaskRCNN.html)

For example:

```text
ROI Features
     ↓
Convolution Layers
     ↓
Upsampling
     ↓
Mask Prediction
     ↓
28 × 28 Mask
```

The mask branch typically produces a separate mask for each object class.

For example, if there are 3 foreground classes:

```text
Mask Output:

[B, 3, 28, 28]
```

The classification branch determines which class is predicted. The corresponding class-specific mask is selected for that detection.

**Important:** The 28×28 mask is a typical Mask R-CNN configuration, not a fixed requirement. The predicted mask is resized and placed within the corresponding detected bounding box in the original image.

---

## 8. Final Output — Instance Segmentation

Now let's combine everything.

![Mask R-CNN instance segmentation output](https://images.openai.com/static-rsc-4/u_tH4Qo00z7SYqG4pY0PFUlLdNJpz_PNOEol3uOEZcAVzH54WBIgFkEZz3P0izA6rCOdDOvpWFAgytvg9-Ch3tAZNo9odu40UAP77s4jGU5zPFjzApNbLWyHQsf3xRcwqwNPd5uLC9n5zOGi_xRlCsGSGgbDNvdDuvihJFir6x8?purpose=inline)

Source: [Instance Segmentation with Mask R-CNN](https://medium.com/data-science/instance-segmentation-with-mask-r-cnn-6e5c4132030b)

Suppose the input image contains:

- Two cars.
- One pedestrian.
- One bicycle.

Mask R-CNN produces:

| Object | Class | Bounding box | Mask |
|---|---|---|---|
| Object 1 | Car | Coordinates | Mask 1 |
| Object 2 | Car | Coordinates | Mask 2 |
| Object 3 | Person | Coordinates | Mask 3 |
| Object 4 | Bicycle | Coordinates | Mask 4 |

Each object receives its own segmentation mask.

This is what makes Mask R-CNN an instance segmentation architecture.

---

## 9. How Is Mask R-CNN Trained?

Mask R-CNN uses multiple losses because it performs multiple prediction tasks.

Its total training loss is commonly expressed as:

$$
L=L_{RPN}+L_{class}+L_{box}+L_{mask}
$$

Let's understand them.

| Loss | Purpose |
|---|---|
| \(L_{RPN}\) | Train objectness and proposal box predictions |
| \(L_{class}\) | Train object classification |
| \(L_{box}\) | Train bounding-box refinement |
| \(L_{mask}\) | Train pixel-level mask prediction |

### Mask Loss

For a particular positive object ROI, the mask branch predicts a binary mask for the ground-truth class.

A common choice is pixel-wise binary cross-entropy.

$$
L_{mask} =
-\frac{1}{N}\sum_{i=1}^{N}
[y_i\log(p_i)+(1-y_i)\log(1-p_i)]
$$

Where:

- \(y_i\) = ground-truth mask pixel.
- \(p_i\) = predicted foreground probability.
- \(N\) = number of mask pixels.

The mask loss is applied to the corresponding ground-truth class channel for each positive ROI.

**Important:** Unlike semantic segmentation, the mask branch does not simply produce one full-image multi-class mask. It predicts an individual mask for each detected object proposal.

---

## 10. Complete Mask R-CNN Architecture — Revision

Let's put all the components together.

![Complete Mask R-CNN architecture](https://images.openai.com/static-rsc-4/m9WkLhnWtaOQP3Mvz0k8A7ZacyVDrb_TXrdhdl-7Z72uJikX2Wyfmuvy-9nq5oEvv3xet-DUm03cxdzZcgdAab3lruB4IvL3VPZoYeuLv4d7A9iDpg5lzmYz1qrFwLvB1NR0tPLFZSiVp6PvlbRoxLfzYd_nnIIQmiBHAYyuizw?purpose=inline)

Source: [Deep Learning Object Detection — Medium](https://krrishrebba.medium.com/part-6-deep-learning-object-detection-revolution-2014-2025-349425f9616b)

```text
                   INPUT IMAGE
                       |
                       ↓
                  CNN BACKBONE
                    (ResNet)
                       |
                       ↓
              FEATURE PYRAMID NETWORK
                    (FPN)
                       |
                       ↓
              REGION PROPOSAL NETWORK
                    (RPN)
                       |
                       ↓
               REGION PROPOSALS
                       |
                       ↓
                    ROI ALIGN
                       |
               ┌───────┼───────┐
               |       |       |
               ↓       ↓       ↓
          CLASSIFICATION BOX    MASK
              HEAD      HEAD   HEAD
               |       |       |
               ↓       ↓       ↓
             Class  Refined  Object
             Label    Box     Mask
               |       |       |
               └───────┼───────┘
                       |
                       ↓
              INSTANCE SEGMENTATION
```

### Training vs Inference

There is an important difference between the two stages.

| Training | Inference |
|---|---|
| Uses images and ground-truth annotations | Uses only input images |
| Computes classification loss | Predicts object classes |
| Computes bounding-box loss | Predicts refined boxes |
| Computes mask loss | Generates individual masks |
| Updates model parameters | Produces final detections |

During inference, confidence filtering and NMS are used to remove unsuitable and duplicate detections.

---

## 11. Mask R-CNN vs Faster R-CNN

This is a very common interview question.

![Faster R-CNN and Mask R-CNN comparison](https://images.openai.com/static-rsc-4/WWUV-_9GpP-IrEvHVULmnouyDGo27Cg_a3EXpyIGrI5boIqQds3V3iOnSUfVfoTBISEC1z9UvXv6fYzQR4GpB9KdZgN4_oTEsElp9JiE4HNf3DO9W6Fa6rbgD3beKomZ6FsWloUcZtMlB-ughwqMZY7zVIIioAjDwrqLHHSB2n0?purpose=inline)

Source: [How Mask R-CNN Works — Esri](https://developers.arcgis.com/python/latest/guide/how-maskrcnn-works/)

| Feature | Faster R-CNN | Mask R-CNN |
|---|---|---|
| Object detection | Yes | Yes |
| Classification | Yes | Yes |
| Bounding-box regression | Yes | Yes |
| Pixel-level masks | No | Yes |
| ROI operation | ROI Pooling in original design | ROI Align |
| Primary task | Object detection | Instance segmentation |

**Remember:** Mask R-CNN extends Faster R-CNN with an additional mask prediction branch and uses ROI Align for better spatial alignment.

---

## 12. Mask R-CNN vs U-Net vs DeepLab

| Feature | U-Net | DeepLabv3+ | Mask R-CNN |
|---|---|---|---|
| Task | Semantic segmentation | Semantic segmentation | Instance segmentation |
| Encoder–decoder | Yes | Yes | Not in the same U-Net-style design |
| Skip connections | Yes | Low-level feature fusion | FPN feature fusion |
| Object proposals | No | No | Yes |
| Bounding boxes | No | No | Yes |
| Individual object masks | No | No | Yes |

---

## 13. Industry Applications

![Mask R-CNN object instance segmentation](https://images.openai.com/static-rsc-4/Hh9Z4o864RPLcj5B0UHhWwsyL316Y0TxANVQBsCeSouGbFNFl9CVGTDxWwYsWrUWuHL-6W3kpbDfYv8P3_YXpiCxeMiEUw9K8LFWoU9tEZ_yPhBtjz7hnZtEOhtaQ-lcvByb4QOB3QWJMQ5P1H6vEz5NvqdrMyREt8bUKFXhE6o?purpose=inline)

![Cell and nuclei instance segmentation](https://images.openai.com/static-rsc-4/2LPJB3JBKnp5ePE0uTLDP2TF0RbNlMQoXCDp9OF1uNnmg0t3h2ZMz71YsPPlgPpEEr-6p33rkVgwLK4micdK2izuf6OXt86Lb1ml162nnPIiwod3MRem0GAmY4MwDOlQuzTORmehkob09xvtcgt0kiJ_4qzkp6zW0uhTm5iXzr8?purpose=inline)

![Fruit instance segmentation](https://images.openai.com/static-rsc-4/-G5pWCUCsJulzeWIN3r1ZvLTGXnLX9MiFipFg5qoFP-O5O5pPOAZx4dOdTC7nJXojWxy7ig2Ug8Y-KjoZLo4iXmw5gOJhrZb3qxQDXIKPyld-emRPTeFCUz6HFYVrPU1yoxTHjrrTkVkoEguMMk5JjWtfAIXGhCsrFojHlGid9o?purpose=inline)

| Industry | Application |
|---|---|
| Autonomous vehicles | Separating individual cars and pedestrians |
| Healthcare | Identifying individual cells and nuclei |
| Manufacturing | Detecting and segmenting individual defects |
| Agriculture | Identifying and counting individual fruits |
| Robotics | Identifying individual objects for manipulation |

---

## 14. Practical Implementation — PyTorch

For your upcoming implementation module, one useful option is the pretrained Mask R-CNN model provided by Torchvision.

It allows you to use an existing model instead of training a complex architecture from scratch.

### Example

```python
import torch

from torchvision.models.detection import (
    maskrcnn_resnet50_fpn,
    MaskRCNN_ResNet50_FPN_Weights
)

# Load pretrained Mask R-CNN
weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT

model = maskrcnn_resnet50_fpn(
    weights=weights
)

# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

print("Device:", device)
```

For inference, Torchvision expects a list of images, each represented as a floating-point tensor of shape `[C, H, W]`, generally with values in the range `[0, 1]`.

```python
# Example input
image = torch.rand(3, 480, 640).to(device)

with torch.inference_mode():
    predictions = model([image])

print(predictions[0].keys())
```

Typical prediction fields:

```text
boxes
labels
scores
masks
```

- `boxes`: predicted bounding boxes.
- `labels`: predicted object classes.
- `scores`: detection confidence scores.
- `masks`: predicted object masks.

The pretrained model uses COCO categories. For a custom dataset, you would need to adapt the prediction heads and train or fine-tune the model using instance-level annotations.

---

## What you should remember 🧠

1. Mask R-CNN is an instance segmentation architecture built on Faster R-CNN.
2. Backbone extracts image features.
3. FPN combines features at multiple resolutions.
4. RPN generates candidate object regions.
5. ROI Align extracts spatially aligned features of fixed dimensions.
6. Three branches perform classification, bounding-box regression and mask prediction.
7. Each detected object receives an individual segmentation mask.
8. Total training loss combines RPN, classification, bounding-box and mask losses.

---
